# Data Understanding — Malware Detection

Nesta fase é realizada a exploração inicial do dataset utilizado no projeto de deteção de malware.

O objetivo é compreender a estrutura dos dados, analisar a distribuição das classes, estudar as principais características, identificar possíveis problemas de qualidade e observar relações entre as variáveis antes da fase de Data Preparation.



In [ ]:
import sys
import subprocess
from pathlib import Path

em_colab = "google.colab" in sys.modules

if em_colab:
    pasta_projeto = Path("/content/IAAC-Projeto")

    if not pasta_projeto.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "https://github.com/diogocarvalho2025056-crypto/IAAC-Projeto.git"
            ],
            check=True
        )
else:
    pasta_projeto = Path.cwd().parent

if str(pasta_projeto) not in sys.path:
    sys.path.insert(0, str(pasta_projeto))

print("Pasta do projeto:")
print(pasta_projeto)

In [ ]:
# --------------------------------------------------
# 1. INSTALAÇÃO DE DEPENDENCIAS DO KAGGLE
# --------------------------------------------------

%pip install kagglehub[pandas-datasets]



In [ ]:
# --------------------------------------------------
# 2. IMPORTS
# --------------------------------------------------

import pandas as pd
import kagglehub

from kagglehub import KaggleDatasetAdapter

from src.eda_utils import (
    resumo_dataset,
    distribuicao_target,
    estatistica_descritiva,
    valores_unicos,
    variaveis_constantes,
    analise_univariada,
    histogramas,
    comparacao_medias,
    comparacao_medianas,
    boxplots_por_classe,
    correlacao_pearson,
    correlacao_spearman,
    heatmap_correlacao,
    correlacao_com_target,
    grafico_correlacao_target,
    pares_alta_correlacao,
    detectar_outliers_iqr
)

## Importação do dataset

O dataset é importado diretamente da plataforma Kaggle, evitando a dependência de uma cópia local para a realização da análise.

É utilizado o ficheiro combinado `Malware_and_benign_recognition.csv`, que contém exemplos classificados como benignos e maliciosos.

In [ ]:
## Importação direta do Kaggle


file_path = "Malware_and_benign_recognition.csv"

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "aashishzzz/malware-detection-dataset-with-new-malware-info",
    file_path
)

print("Dataset importado com sucesso.")

In [ ]:
# --------------------------------------------------
# 3. CONFIRMAR IMPORTAÇÃO E VISUALIZAÇÃO INICIAL
# --------------------------------------------------

print("Dimensão do dataset:")
print(df.shape)

print("\nPrimeiros 5 registos:")
display(df.head())

## Análise geral do dataset

Nesta etapa são analisadas as dimensões do conjunto de dados, tipos de variáveis, existência de valores em falta e possíveis registos duplicados.

In [ ]:
resumo_dataset(df)

In [ ]:
distribuicao_target(
    df,
    "Malicious"
)

### Distribuição das classes

A distribuição da variável `Malicious` é praticamente equilibrada entre as duas classes.

O valor `0` corresponde a ficheiros benignos e o valor `1` a ficheiros maliciosos.

Este equilíbrio reduz a necessidade de aplicar técnicas específicas de correção de desequilíbrio de classes nesta fase.

In [ ]:
print("\n--- ESTATÍSTICA DESCRITIVA ---")
estatistica_descritiva(df)

In [ ]:
print("\n--- NÚMERO DE VALORES ÚNICOS ---")
valores_unicos(df)

In [ ]:
print("\n--- VARIÁVEIS CONSTANTES ---")
constantes = variaveis_constantes(df)
print(constantes)

### Qualidade inicial dos dados

A análise inicial não identificou valores em falta.

Foi identificada a variável `SuspiciousSectionNames` como constante, apresentando sempre o mesmo valor. Uma variável sem variabilidade não fornece informação útil para distinguir as classes e deverá ser considerada para remoção durante a Data Preparation.

Também foi identificada redundância entre `NumberOfSections` e `SectionCount`, que apresentam a mesma informação.

In [ ]:
features = [
    "AvgSectionEntropy",
    "ImportedDLLs",
    "ImportedFunctions",
    "SizeOfCode",
    "SizeOfImage",
    "OverlaySize"
]

print("Features selecionadas para análise:")
print(features)

## Análise Univariada

A análise univariada permite observar individualmente a distribuição e as principais medidas estatísticas de cada característica.

Foram selecionadas algumas variáveis representativas das propriedades estruturais dos ficheiros PE.

In [ ]:
analise_univariada(
    df,
    features
)

In [ ]:
histogramas(
    df,
    features
)

### Observações

Algumas variáveis apresentam distribuições bastante assimétricas.

`SizeOfCode`, `SizeOfImage` e `OverlaySize` apresentam valores extremos e diferenças consideráveis entre média e mediana.

Estas observações serão consideradas na fase de Data Preparation, sem remover automaticamente os valores extremos, uma vez que determinadas características pouco frequentes podem representar informação relevante na identificação de malware.

## Análise Bivariada

Nesta etapa são comparados os valores das características entre as duas classes da variável alvo: ficheiros benignos e ficheiros maliciosos.

In [ ]:
medias = comparacao_medias(
    df,
    "Malicious",
    features
)

print("--- MÉDIAS POR CLASSE ---")
display(medias)

In [ ]:
medianas = comparacao_medianas(
    df,
    "Malicious",
    features
)

print("--- MEDIANAS POR CLASSE ---")
display(medianas)

In [ ]:
boxplots_por_classe(
    df,
    "Malicious",
    features,
    ["Benigno", "Malware"]
)

### Observações da análise bivariada

Foram observadas diferenças entre ficheiros benignos e maliciosos em várias características.

Os ficheiros maliciosos apresentam, em média, valores superiores de `AvgSectionEntropy` e `SizeOfImage`.

Por outro lado, os ficheiros benignos apresentam valores médios superiores de `ImportedDLLs` e `ImportedFunctions`.

A variável `OverlaySize` apresenta valores extremos, pelo que a média isoladamente deve ser interpretada com precaução.

## Análise de Correlação

Foram utilizados os coeficientes de Pearson e Spearman.

Pearson permite observar principalmente relações lineares entre as variáveis, enquanto Spearman permite identificar relações monotónicas através da ordenação dos valores.

In [ ]:
pearson = correlacao_pearson(
    df,
    coluna_texto="File"
)

heatmap_correlacao(
    pearson,
    "Matriz de Correlação de Pearson"
)

In [ ]:
spearman = correlacao_spearman(
    df,
    coluna_texto="File"
)

heatmap_correlacao(
    spearman,
    "Matriz de Correlação de Spearman"
)

In [ ]:
# Correlação com target — Pearson

corr_target_pearson = correlacao_com_target(
    pearson,
    "Malicious"
)

display(corr_target_pearson)

grafico_correlacao_target(
    corr_target_pearson,
    "Correlação de Pearson com Malicious"
)

In [ ]:
# Correlação Spearman com target

corr_target_spearman = correlacao_com_target(
    spearman,
    "Malicious"
)

display(corr_target_spearman)

grafico_correlacao_target(
    corr_target_spearman,
    "Correlação de Spearman com Malicious"
)

In [ ]:
# Features muito correlacionadas entre si - Pearson

print("--- PARES COM CORRELAÇÃO DE PEARSON >= 0.80 ---")

pares_pearson = pares_alta_correlacao(
    pearson,
    limite=0.80
)

for coluna1, coluna2, valor in pares_pearson:
    print(
        coluna1,
        "<->",
        coluna2,
        ":",
        round(valor, 3)
    )

In [ ]:
# Features muito correlacionadas entre si - Spearman

print("--- PARES COM CORRELAÇÃO DE SPEARMAN >= 0.80 ---")

pares_spearman = pares_alta_correlacao(
    spearman,
    limite=0.80
)

for coluna1, coluna2, valor in pares_spearman:
    print(
        coluna1,
        "<->",
        coluna2,
        ":",
        round(valor, 3)
    )

## Análise de Outliers

Os possíveis valores extremos são identificados através do método do intervalo interquartil (IQR).

Nesta fase os outliers são apenas identificados e analisados. Não serão automaticamente removidos, uma vez que valores estruturais pouco frequentes podem representar características relevantes de determinados ficheiros maliciosos.

In [ ]:
outliers = detectar_outliers_iqr(
    df,
    features
)

print(outliers)

## Conclusão

A fase de Data Understanding permitiu conhecer a estrutura e as principais características do dataset.

O conjunto analisado contém 1.156 registos e apresenta uma distribuição praticamente equilibrada entre ficheiros benignos e maliciosos.

Não foram identificados valores em falta. Foram, contudo, identificadas algumas características que deverão ser consideradas durante a preparação dos dados, nomeadamente:

- `File`, por funcionar como identificador do ficheiro;
- `SuspiciousSectionNames`, por ser uma variável constante;
- redundância entre `NumberOfSections` e `SectionCount`;
- diferenças significativas de escala entre várias características;
- presença de valores extremos;
- existência de correlações elevadas entre algumas variáveis.

As análises de Pearson e Spearman também mostraram diferentes níveis de associação entre algumas características e a variável alvo `Malicious`.

Estas conclusões serão utilizadas na fase seguinte, Data Preparation, para definir as operações de limpeza, seleção de características, divisão dos dados e standardização.